In [ ]:
%pip install -q pandas numpy openai jsonschema tenacity

In [ ]:
import json
import pandas as pd
import numpy as np
from openai import OpenAI
from tenacity import retry, wait_exponential_jitter, stop_after_attempt, retry_if_exception_type
from jsonschema import validate, ValidationError

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
DATA_PATH = "/content/drive/MyDrive/Special Topics LLM Project/sample_ecommerce_kpi_data.csv"
MODEL = "gpt-4.1-nano"

In [ ]:
DF = pd.read_csv(DATA_PATH)
DF.head()

,date,region,device_type,product_category,traffic,conversion_rate,orders,avg_order_value,revenue,marketing_spend
0,2024-01-01,North,desktop,Electronics,590,0.0382,22,162.41,3573.12,734.03
1,2024-01-01,North,desktop,Clothing,568,0.0400,22,85.11,1872.31,1562.11
2,2024-01-01,North,desktop,Home,628,0.0327,20,70.34,1406.85,956.36
3,2024-01-01,North,desktop,Sports,601,0.0391,23,59.76,1374.51,709.24
4,2024-01-01,North,mobile,Electronics,874,0.0360,31,130.94,4059.02,799.51


In [ ]:
if "date" not in DF.columns:
    raise ValueError(f"'date' column not found. Columns: {DF.columns.tolist()}")
DF["date"] = pd.to_datetime(DF["date"], errors="coerce")
DF = DF.dropna(subset=["date"]).copy()

Dimensions (non-numeric columns) and metric keys (numeric columns)
Purpose: Prevent the model from hallucinating columns or passing invalid parameters.

In [ ]:
ALL_COLUMNS = DF.columns.tolist()

DIMENSION_KEYS = [
    c for c in ALL_COLUMNS
    if c != "date" and not pd.api.types.is_numeric_dtype(DF[c])
]
METRIC_KEYS = [
    c for c in ALL_COLUMNS
    if c != "date" and pd.api.types.is_numeric_dtype(DF[c])
]

In [ ]:
print("DIMENSION_KEYS:", DIMENSION_KEYS)
print("METRIC_KEYS:", METRIC_KEYS)
print("DATE RANGE:", DF["date"].min().date(), "to", DF["date"].max().date())

DIMENSION_KEYS: ['region', 'device_type', 'product_category']
METRIC_KEYS: ['traffic', 'conversion_rate', 'orders', 'avg_order_value', 'revenue', 'marketing_spend']
DATE RANGE: 2024-01-01 to 2024-05-31


In [ ]:
def detect_anomaly(
    metric: str,
    date_col: str = "date",
    group_by: dict | None = None,    # e.g., device_type or region
    start_date: str | None = None,
    end_date: str | None = None,
    method: str = "zscore",
    z_threshold: float = 2.5,
    min_periods: int = 14
):
    """
    Local anomaly detection tool:
    - Filters by optional date window and optional dimension constraints.
    - Computes daily metric series (sum) and flags points with large z-score vs rolling window.
    - Returns a detected change window and summary stats.
    """
    try:
        if metric not in METRIC_KEYS:
            return {"success": False, "error": f"Invalid metric '{metric}'", "allowed_metrics": METRIC_KEYS}

        dff = DF.copy()

        # Optional date filter
        if start_date:
            sd = pd.to_datetime(start_date, errors="coerce")
            if pd.isna(sd):
                return {"success": False, "error": "Invalid start_date. Expected YYYY-MM-DD."}
            dff = dff[dff[date_col] >= sd]

        if end_date:
            ed = pd.to_datetime(end_date, errors="coerce")
            if pd.isna(ed):
                return {"success": False, "error": "Invalid end_date. Expected YYYY-MM-DD."}
            dff = dff[dff[date_col] <= ed]

        # Optional dimension constraints (case-insensitive for strings)
        if group_by:
            for k, v in group_by.items():
                if k not in DIMENSION_KEYS:
                    return {"success": False, "error": f"Invalid dimension key '{k}'", "allowed_dimensions": DIMENSION_KEYS}
                col = dff[k]
                if col.dtype == "object":
                    dff = dff[col.astype(str).str.lower() == str(v).lower()]
                else:
                    dff = dff[col == v]

        if dff.empty:
            return {"success": False, "error": "No data after filters. Check date range / dimensions."}

        # Build daily series
        daily = dff.groupby(dff[date_col].dt.date)[metric].sum().reset_index()
        daily.columns = ["day", metric]
        daily["day"] = pd.to_datetime(daily["day"])
        daily = daily.sort_values("day").reset_index(drop=True)

        if len(daily) < min_periods:
            return {"success": False, "error": f"Not enough days for anomaly detection (need >= {min_periods}).", "days": int(len(daily))}

        # Rolling z-score of daily values
        window = min_periods
        rolling_mean = daily[metric].rolling(window=window, min_periods=window).mean()
        rolling_std = daily[metric].rolling(window=window, min_periods=window).std(ddof=0)

        z = (daily[metric] - rolling_mean) / rolling_std
        daily["zscore"] = z

        anomalies = daily[(daily["zscore"].abs() >= z_threshold) & daily["zscore"].notna()].copy()

        if anomalies.empty:
            return {
                "success": True,
                "metric": metric,
                "found_anomaly": False,
                "note": "No anomalies detected at the chosen threshold.",
                "params": {"method": method, "z_threshold": z_threshold, "window": window},
            }

        first = anomalies.iloc[0]
        # approximate baseline from prior rolling window
        idx = anomalies.index[0]
        baseline_slice = daily.iloc[max(0, idx-window):idx]
        baseline_mean = float(baseline_slice[metric].mean()) if len(baseline_slice) else None

        return {
            "success": True,
            "metric": metric,
            "found_anomaly": True,
            "anomaly_date": first["day"].strftime("%Y-%m-%d"),
            "zscore": float(first["zscore"]),
            "baseline_mean": baseline_mean,
            "anomaly_value": float(first[metric]),
            "params": {"method": method, "z_threshold": z_threshold, "window": window},
        }

    except Exception as e:
        return {"success": False, "error": str(e)}

Query KPI Data
Purpose: The model calls it when it needs a real data slice.
Inputs (tool arguments):
- date_start, date_end: the anomaly window
- dimensions: filters like {"region":"West","device_type":"Mobile"}
- metrics: numeric columns to compute/return
- agg: whether to sum or average metrics over the slice
- limit: bounds the preview size (cost/latency control)

Outputs (returned to the model):
- rows_returned: how many rows matched
- aggregates: deterministic totals/means for the metrics
- preview: first N rows (bounded) for context

In [ ]:
def query_kpi_data(
    source: str,
    date_start: str,
    date_end: str,
    dimensions: dict,
    metrics: list,
    agg: str = "sum",
    include_diagnostics: bool = True
):
    """
    Local data query tool (privacy-safe): returns aggregates only.
    """
    try:
        ds = pd.to_datetime(date_start, errors="coerce")
        de = pd.to_datetime(date_end, errors="coerce")
        if pd.isna(ds) or pd.isna(de):
            return {"success": False, "error": "Invalid date_start/date_end; expected YYYY-MM-DD."}

        # Validate metrics
        for m in metrics:
            if m not in METRIC_KEYS:
                return {"success": False, "error": f"Invalid metric '{m}'", "allowed_metrics": METRIC_KEYS}

        base = DF[(DF["date"] >= ds) & (DF["date"] <= de)].copy()

        dff = base
        for k, v in dimensions.items():
            if k not in DIMENSION_KEYS:
                return {"success": False, "error": f"Invalid dimension key '{k}'", "allowed_dimensions": DIMENSION_KEYS}
            col = dff[k]
            if col.dtype == "object":
                dff = dff[col.astype(str).str.lower() == str(v).lower()]
            else:
                dff = dff[col == v]

        if dff.empty:
            out = {
                "success": False,
                "error": "No rows matched filters in the requested date range.",
                "requested": {"date_start": date_start, "date_end": date_end, "dimensions": dimensions, "metrics": metrics},
            }
            if include_diagnostics:
                diag = {
                    "dataset_date_min": DF["date"].min().strftime("%Y-%m-%d"),
                    "dataset_date_max": DF["date"].max().strftime("%Y-%m-%d"),
                }
                for k in dimensions.keys():
                    if k in base.columns:
                        diag[f"available_{k}_in_window"] = sorted(base[k].dropna().astype(str).unique().tolist())[:50]
                out["diagnostics"] = diag
            return out

        if agg == "sum":
            aggregates = {m: float(dff[m].sum()) for m in metrics}
        elif agg == "mean":
            aggregates = {m: float(dff[m].mean()) for m in metrics}
        else:
            return {"success": False, "error": "agg must be 'sum' or 'mean'."}

        return {"success": True, "rows_returned": int(len(dff)), "aggregates": aggregates}

    except Exception as e:
        return {"success": False, "error": str(e)}

Tool
compute_kpi_stats
Purpose: This is the “calculator tool” for precise arithmetic that LLMs often get wrong.
When the model should call it: Anytime it must compute percent change, ratios, deltas, etc.
Output: A deterministic number (no hallucination).

In [ ]:
def compute_kpi_stats(metric: str, baseline_value: float, current_value: float):
    try:
        if metric == "percent_change":
            if baseline_value == 0:
                return {"success": False, "error": "baseline_value cannot be 0"}
            return {"success": True, "percent_change": (current_value - baseline_value) / baseline_value}
        return {"success": False, "error": f"Unsupported metric '{metric}'"}
    except Exception as e:
        return {"success": False, "error": str(e)}

Tool Schemas
Purpose: Tell the LLM exactly how to call tools and with what parameters, reducing tool-call failures

In [ ]:
TOOL_IMPL = {
    "detect_anomaly": detect_anomaly,
    "query_kpi_data": query_kpi_data,
    "compute_kpi_stats": compute_kpi_stats,
}


In [ ]:
TOOLS = [
    {
        "type": "function",
        "name": "detect_anomaly",
        "description": (
            "Detect anomalies in a daily KPI metric using a rolling z-score method. "
            "Use to identify the anomaly start date and baseline context."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "metric": {"type": "string", "enum": METRIC_KEYS},
                "date_col": {"type": "string", "default": "date"},
                "group_by": {"type": "object", "description": f"Optional dimension filters; keys allowed: {DIMENSION_KEYS}"},
                "start_date": {"type": "string", "description": "YYYY-MM-DD", "default": None},
                "end_date": {"type": "string", "description": "YYYY-MM-DD", "default": None},
                "method": {"type": "string", "enum": ["zscore"], "default": "zscore"},
                "z_threshold": {"type": "number", "minimum": 1.0, "maximum": 10.0, "default": 2.5},
                "min_periods": {"type": "integer", "minimum": 7, "maximum": 60, "default": 14},
            },
            "required": ["metric"],
            "additionalProperties": False
        }
    },
    {
        "type": "function",
        "name": "query_kpi_data",
        "description": "Query KPI aggregates over a date range with dimension filters (privacy-safe aggregates only).",
        "parameters": {
            "type": "object",
            "properties": {
                "source": {"type": "string"},
                "date_start": {"type": "string"},
                "date_end": {"type": "string"},
                "dimensions": {"type": "object", "description": f"Keys allowed: {DIMENSION_KEYS}"},
                "metrics": {"type": "array", "items": {"type": "string", "enum": METRIC_KEYS}},
                "agg": {"type": "string", "enum": ["sum", "mean"], "default": "sum"},
                "include_diagnostics": {"type": "boolean", "default": True},
            },
            "required": ["source", "date_start", "date_end", "dimensions", "metrics"],
            "additionalProperties": False
        }
    },
    {
        "type": "function",
        "name": "compute_kpi_stats",
        "description": "Compute deterministic KPI math (percent change).",
        "parameters": {
            "type": "object",
            "properties": {
                "metric": {"type": "string", "enum": ["percent_change"]},
                "baseline_value": {"type": "number"},
                "current_value": {"type": "number"},
            },
            "required": ["metric", "baseline_value", "current_value"],
            "additionalProperties": False
        }
    }
]

TOOL_SCHEMAS = {t["name"]: t["parameters"] for t in TOOLS}

Prevent transient failures from breaking the pipeline

In [ ]:
@retry(
    wait=wait_exponential_jitter(initial=0.5, max=8),
    stop=stop_after_attempt(4),
    retry=retry_if_exception_type(Exception),
)
def call_model(*, input, tools=None, previous_response_id=None, tool_choice="auto", max_output_tokens=900):
    return client.responses.create(
        model=MODEL,
        input=input,
        tools=tools,
        previous_response_id=previous_response_id,
        tool_choice=tool_choice,
        max_output_tokens=max_output_tokens,
    )

Tool Calling
1. Send user prompt + tool schemas
2. model chooses a tool call (JSON)
3. code executes tool
4. send tool results back
5. model produces final answer

In [ ]:
def run_agent(system_prompt: str, user_prompt: str, max_rounds: int = 8, debug: bool = True) -> str:
    response = call_model(
        input=[{"role":"system","content":system_prompt},{"role":"user","content":user_prompt}],
        tools=TOOLS,
        tool_choice="auto",
        max_output_tokens=900,
    )

    had_failure = False

    for r in range(max_rounds):
        output_items = getattr(response, "output", [])
        call_items = [o for o in output_items if getattr(o, "type", None) == "function_call"]

        if debug:
            print(f"[round {r}] response.output types:", [getattr(o, "type", None) for o in output_items])

        if not call_items:
            return response.output_text or ""

        function_outputs = []
        all_success = True

        for call in call_items:
            name = call.name
            args = call.arguments if isinstance(call.arguments, dict) else json.loads(call.arguments)

            # Validate against schema
            if name in TOOL_SCHEMAS:
                try:
                    validate(instance=args, schema=TOOL_SCHEMAS[name])
                except ValidationError as ve:
                    result = {"success": False, "error": "Schema validation failed", "details": str(ve), "tool": name, "args": args}
                    all_success = False
                    function_outputs.append({"type":"function_call_output","call_id":call.call_id,"output":json.dumps(result)})
                    continue

            # Execute tool
            try:
                if debug:
                    print("  TOOL CALL:", name, args)
                result = TOOL_IMPL[name](**args)
                if debug:
                    print("  TOOL RESULT:", result)
            except Exception as e:
                result = {"success": False, "error": str(e), "tool": name, "args": args}

            if not result.get("success", False):
                all_success = False

            function_outputs.append({
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": json.dumps(result),
            })

        # - If tools succeeded -> force final synthesis
        # - If tools failed -> allow ONE correction attempt, then force synthesis with diagnostics
        if all_success:
            next_tool_choice = "none"
        else:
            if had_failure:
                next_tool_choice = "none"
            else:
                next_tool_choice = "auto"
                had_failure = True

        response = call_model(
            previous_response_id=response.id,
            input=function_outputs,
            tools=TOOLS,
            tool_choice=next_tool_choice,
            max_output_tokens=900,
        )

    return "Stopped after max tool rounds."


Testing tool selection accuracy + parameter

In [ ]:
tests = [
  {"id":"math", "prompt":"Revenue went from 100 to 85. Compute percent change.", "expected_tool":"compute_kpi_stats"},
  {"id":"data", "prompt":"Pull revenue and orders for West+Mobile from 2024-04-01 to 2024-04-30.", "expected_tool":"query_kpi_data"},
  {"id":"no_tool", "prompt":"Write an exec summary using only provided findings; do not fetch more data.", "expected_tool":None},
]

In [ ]:
COLUMNS = "date, revenue, orders, traffic, conversion_rate, avg_order_value, region, device_type, marketing_spend, product_category"
ANOMALY = "Revenue dropped 15% starting April 20, 2024 (p < 0.01)."
FINDINGS = """Primary driver: Mobile conversion_rate dropped 22% (r = 0.71, p < 0.01).
Secondary factor: marketing_spend in West region decreased 35%.
Ruled out: avg_order_value (no significant change).
Ruled out: product_category mix (stable)."""

instruction_system = """You are a KPI root-cause analysis assistant for an e-commerce business.
Your task is to generate executive-facing explanations of KPI anomalies.
You must ONLY use the facts explicitly provided below.
Do NOT invent metrics, variables, or causes.
If information is missing, explicitly state the uncertainty and propose analytical checks."""

instruction_user = f"""Dataset columns: {COLUMNS}

Detected anomaly: {ANOMALY}

Statistical findings (precomputed):
{FINDINGS}

Constraints:
- Base all claims strictly on the findings above.
- Clearly separate EVIDENCE from HYPOTHESES.
- Do not introduce unsupported causes.

RESPONSE FORMAT:
1) Executive summary (2–3 sentences)
2) Key evidence (bullets; use only provided numbers)
3) Hypotheses (bullets; clearly labeled)
4) Recommended next analyses (bullets)
"""

Mode A: precomputed anomaly + finidngs, no tool calls expected; pure synthesis

In [ ]:
print("\n===== MODE A: Synthesize provided anomaly + findings =====\n")
final_text_A = run_agent(instruction_system, instruction_user, debug=True)
print("\n=== FINAL ANSWER (A) ===\n")
print(final_text_A)


===== MODE A: Synthesize provided anomaly + findings =====

[round 0] response.output types: ['message']

=== FINAL ANSWER (A) ===

1) **Executive Summary**  
The dataset indicates a significant 15% drop in revenue beginning April 20, 2024, primarily associated with a decline in mobile conversion rates. Additionally, a decrease in marketing spend in the West region may have contributed to this revenue decline.

2) **Key Evidence**  
- Revenue dropped 15% starting April 20, 2024 (p < 0.01).  
- Mobile conversion rate decreased by 22% (r = 0.71, p < 0.01).  
- Marketing spend in the West region decreased by 35%.  
- No significant change observed in avg_order_value.  
- Product category mix remained stable.

3) **Hypotheses**  
- The decline in mobile conversion rate is likely a primary driver of the revenue decrease.  
- The reduction in marketing spend in the West region may have negatively impacted overall revenue.  

4) **Recommended Next Analyses**  
- Investigate whether other reg

Mode B: ask agent to detect anomalies first

In [ ]:
system_prompt_B = f"""
You are a KPI root-cause analysis assistant for an e-commerce business.

Goal:
1) Detect an anomaly in revenue using tools.
2) Summarize the anomaly.
3) Provide executive-ready output.

Rules:
- Use tools for detection and any necessary aggregates.
- After tools succeed, produce the final response using:
  1) Executive summary
  2) Key evidence
  3) Hypotheses
  4) Recommended next analyses
- Do not invent numbers. Only use tool outputs.
"""

user_prompt_B = """
Detect whether revenue has a significant anomaly in April 2024.
If you find an anomaly, report the first anomaly date and compute the percent change vs baseline_mean if available.
Then produce the executive-ready output in the required format.
"""

In [ ]:
print("\n===== MODE B: Tool-assisted anomaly detection + executive output =====\n")
final_text_B = run_agent(system_prompt_B, user_prompt_B, debug=True)
print("\n=== FINAL ANSWER (B) ===\n")
print(final_text_B)


===== MODE B: Tool-assisted anomaly detection + executive output =====

[round 0] response.output types: ['function_call', 'function_call']
  TOOL CALL: detect_anomaly {'metric': 'revenue', 'start_date': '2024-04-01', 'end_date': '2024-04-30', 'z_threshold': 2.5, 'min_periods': 14}
  TOOL RESULT: {'success': True, 'metric': 'revenue', 'found_anomaly': False, 'note': 'No anomalies detected at the chosen threshold.', 'params': {'method': 'zscore', 'z_threshold': 2.5, 'window': 14}}
[round 1] response.output types: ['message']

=== FINAL ANSWER (B) ===

No significant anomaly was detected in revenue for April 2024 at the standard threshold. Would you like me to explore other thresholds or analyze a different aspect?


**4. Test and evaluate tool selection accuracy and parameter correctness during execution.**

In [ ]:
def extract_function_calls(resp):
    """
    Returns a list of dicts: [{"name":..., "args":...}, ...]
    from resp.output items of type "function_call".
    """
    calls = []
    for o in getattr(resp, "output", []):
        if getattr(o, "type", None) == "function_call":
            args = o.arguments if isinstance(o.arguments, dict) else json.loads(o.arguments)
            calls.append({"name": o.name, "args": args})
    return calls

In [ ]:
def one_shot_tool_decision(system_prompt: str, user_prompt: str):
    """
    One model call with tools enabled, so we can evaluate tool selection + params
    WITHOUT executing tools. This isolates the model's tool choice behavior.
    """
    return call_model(
        input=[{"role":"system","content":system_prompt},{"role":"user","content":user_prompt}],
        tools=TOOLS,
        tool_choice="auto",
        max_output_tokens=300
    )

In [ ]:
def schema_valid(tool_name: str, args: dict) -> bool:
    if tool_name not in TOOL_SCHEMAS:
        return False
    try:
        validate(instance=args, schema=TOOL_SCHEMAS[tool_name])
        return True
    except ValidationError:
        return False


Parameter correctness checks (semantic checks beyond schema)

In [ ]:
def semantic_checks(tool_name: str, args: dict):
    if tool_name == "query_kpi_data":
        # Dates exist and ordered
        try:
            ds = pd.to_datetime(args.get("date_start"))
            de = pd.to_datetime(args.get("date_end"))
            if ds > de:
                return False, "date_start > date_end"
        except Exception:
            return False, "invalid date format"

        # Must request allowed metrics
        metrics = args.get("metrics", [])
        if not metrics:
            return False, "metrics empty"
        bad = [m for m in metrics if m not in METRIC_KEYS]
        if bad:
            return False, f"invalid metrics {bad}"

        # If user asked for region/device_type, ensure dimensions include those keys
        dims = args.get("dimensions", {})
        # (We can't infer user request here; in tests we encode expected required keys.)
        return True, "ok"

    if tool_name == "detect_anomaly":
        # metric must be one of numeric metrics
        m = args.get("metric")
        if m not in METRIC_KEYS:
            return False, "invalid metric"
        return True, "ok"

    if tool_name == "compute_kpi_stats":
        if args.get("metric") != "percent_change":
            return False, "unsupported calculator metric"
        if args.get("baseline_value") == 0:
            return False, "baseline_value==0"
        return True, "ok"

    # Unknown tool
    return False, "unknown tool"


In [ ]:
TESTS = [
    {
        "id": "T1_data_slice",
        "system": "You are a KPI RCA assistant. Use tools only when needed.",
        "user": "Pull revenue and orders for region='West' and device_type='Mobile' from 2024-04-01 to 2024-04-30.",
        "expected_tools": ["query_kpi_data"],
        "expected_dims_keys": ["region", "device_type"],
    },
    {
        "id": "T2_anomaly_detect",
        "system": "You are a KPI RCA assistant. Use tools to detect anomalies when asked.",
        "user": "Detect whether revenue has an anomaly in April 2024 and return the first anomaly date.",
        "expected_tools": ["detect_anomaly"],
    },
    {
        "id": "T3_calculate",
        "system": "You are a KPI RCA assistant. Use tools for precise calculations.",
        "user": "Revenue went from 100 to 85. Compute percent change.",
        "expected_tools": ["compute_kpi_stats"],
    },
    {
        "id": "T4_no_tool_exec_summary",
        "system": "You are a KPI RCA assistant. If the user provides findings, do not fetch more data.",
        "user": """Detected anomaly: Revenue dropped 15% starting April 20, 2024 (p < 0.01).
Findings: Mobile conversion_rate dropped 22% (r=0.71, p<0.01).
Write an executive summary using ONLY these findings.""",
        "expected_tools": [],  # should not call tools
    },
    {
        "id": "T5_multitool_parallel_like",
        "system": "You are a KPI RCA assistant.",
        "user": "Detect anomaly in revenue in April 2024 and compute percent change versus baseline mean if available.",
        "expected_tools": ["detect_anomaly"],  # may also call compute_kpi_stats later in full execution, but first step should be detect_anomaly
    },
]


Evaluation: tool selection accuracy + parameter correctness

In [ ]:
rows = []

for t in TESTS:
    resp = one_shot_tool_decision(t["system"], t["user"])
    calls = extract_function_calls(resp)

    # Tool selection
    if not t["expected_tools"]:
        tool_selected_correct = (len(calls) == 0)
        chosen_tool = None if not calls else calls[0]["name"]
        args = {} if not calls else calls[0]["args"]
    else:
        chosen_tool = calls[0]["name"] if calls else None
        args = calls[0]["args"] if calls else {}
        tool_selected_correct = (chosen_tool in t["expected_tools"])

    # Schema validity + semantic validity
    if chosen_tool:
        s_valid = schema_valid(chosen_tool, args)
        sem_ok, sem_note = semantic_checks(chosen_tool, args)

        # Additional test-specific semantic check: required dimensions present
        if chosen_tool == "query_kpi_data" and "expected_dims_keys" in t:
            dims = args.get("dimensions", {}) or {}
            missing = [k for k in t["expected_dims_keys"] if k not in dims]
            if missing:
                sem_ok = False
                sem_note = f"missing dimensions keys: {missing}"
    else:
        s_valid = True
        sem_ok = True
        sem_note = "no tool call"

    rows.append({
        "test_id": t["id"],
        "expected_tools": ",".join(t["expected_tools"]) if t["expected_tools"] else "(none)",
        "chosen_tool": chosen_tool or "(none)",
        "tool_selection_correct": tool_selected_correct,
        "schema_valid": s_valid,
        "semantic_valid": sem_ok,
        "notes": sem_note
    })

results_df = pd.DataFrame(rows)
results_df

,test_id,expected_tools,chosen_tool,tool_selection_correct,schema_valid,semantic_valid,notes
0,T1_data_slice,query_kpi_data,query_kpi_data,True,False,False,"missing dimensions keys: ['region', 'device_ty..."
1,T2_anomaly_detect,detect_anomaly,detect_anomaly,True,True,True,ok
2,T3_calculate,compute_kpi_stats,compute_kpi_stats,True,True,True,ok
3,T4_no_tool_exec_summary,(none),(none),True,True,True,no tool call
4,T5_multitool_parallel_like,detect_anomaly,detect_anomaly,True,True,True,ok


In [ ]:
tool_selection_accuracy = results_df["tool_selection_correct"].mean()
schema_valid_rate = results_df["schema_valid"].mean()
semantic_valid_rate = results_df["semantic_valid"].mean()

print("Tool selection accuracy:", round(tool_selection_accuracy, 3))
print("Schema validity rate:", round(schema_valid_rate, 3))
print("Semantic parameter correctness rate:", round(semantic_valid_rate, 3))

Tool selection accuracy: 1.0
Schema validity rate: 0.8
Semantic parameter correctness rate: 0.8
